# Import Data + Preprocessing


In [1]:
#import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import re
import math
from collections import Counter, defaultdict
%matplotlib inline
import random
import itertools
from sentence_transformers import SentenceTransformer, util

In [2]:
import gdown
import pandas as pd

file_id = "1kSQIo1kkDb53Mg4hbBehsEk57u9mjzkF"
url = f"https://drive.google.com/uc?id={file_id}"

df = pd.read_csv(url)
print(df.shape)
df.head()

(45466, 24)


/tmp/ipykernel_2441/2522543376.py:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [3]:
#sắp xếp lại thứ tự các cột

new_order = [
    'id', 'imdb_id', 'title', 'original_title', 'release_date', 'runtime',
    'genres', 'tagline', 'overview', 'budget', 'revenue', 'production_companies',
    'production_countries', 'popularity', 'vote_average', 'vote_count',
    'original_language', 'spoken_languages', 'status', 'adult', 'video',
    'homepage', 'poster_path', 'belongs_to_collection'
]

#reindex lại DataFrame theo thứ tự mới
df = df[new_order]

#check lại
print(df.head())

      id    imdb_id                        title               original_title  \
0    862  tt0114709                    Toy Story                    Toy Story   
1   8844  tt0113497                      Jumanji                      Jumanji   
2  15602  tt0113228             Grumpier Old Men             Grumpier Old Men   
3  31357  tt0114885            Waiting to Exhale            Waiting to Exhale   
4  11862  tt0113041  Father of the Bride Part II  Father of the Bride Part II   

  release_date  runtime                                             genres  \
0   1995-10-30     81.0  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1   1995-12-15    104.0  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2   1995-12-22    101.0  [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...   
3   1995-12-22    127.0  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   
4   1995-02-10    106.0                     [{'id': 35, 'name': 'Comedy'}]   

                                            

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     45466 non-null  object 
 1   imdb_id                45449 non-null  object 
 2   title                  45460 non-null  object 
 3   original_title         45466 non-null  object 
 4   release_date           45379 non-null  object 
 5   runtime                45203 non-null  float64
 6   genres                 45466 non-null  object 
 7   tagline                20412 non-null  object 
 8   overview               44512 non-null  object 
 9   budget                 45466 non-null  object 
 10  revenue                45460 non-null  float64
 11  production_companies   45463 non-null  object 
 12  production_countries   45463 non-null  object 
 13  popularity             45461 non-null  object 
 14  vote_average           45460 non-null  float64
 15  vo

In [5]:
df.shape

(45466, 24)

In [ ]:
print(df.dtypes)

id                        object
imdb_id                   object
title                     object
original_title            object
release_date              object
runtime                  float64
genres                    object
tagline                   object
overview                  object
budget                    object
revenue                  float64
production_companies      object
production_countries      object
popularity                object
vote_average             float64
vote_count               float64
original_language         object
spoken_languages          object
status                    object
adult                     object
video                     object
homepage                  object
poster_path               object
belongs_to_collection     object
dtype: object


In [6]:
#check duplicate

total_duplicates = df.duplicated().sum()
print(f"Số dòng bị trùng lặp hoàn toàn: {total_duplicates}")

#xóa dup, chỉ giữ lại dòng đầu tiên, đè trực tiếp lên df cũ (inplace=True)
df.drop_duplicates(subset=['id'], keep='first', inplace=True)

# Kiểm tra lại số lượng dòng sau khi xóa
print(df.shape)

Số dòng bị trùng lặp hoàn toàn: 13
(45436, 24)


In [7]:
# thống kê Min, Max, Mean, Median của các cột số
print(df[['runtime', 'budget', 'revenue', 'vote_average']].describe())

# min dính 0 nhiều, max dính số quá dài (1256 phút ~ 21 tiếng), median hợp lý (1 fim ~ 1h35')
# min, 25%, 50%, 75% revenue đều bằng 0 -> thiếu dữ liệu
# vote average min = 0 -> chưa ai xem hoặc k có vote

           runtime       revenue  vote_average
count  45173.00000  4.543000e+04  45430.000000
mean      94.12430  1.121288e+07      5.618329
std       38.41554  6.435213e+07      1.924139
min        0.00000  0.000000e+00      0.000000
25%       85.00000  0.000000e+00      5.000000
50%       95.00000  0.000000e+00      6.000000
75%      107.00000  0.000000e+00      6.800000
max     1256.00000  2.787965e+09     10.000000


In [8]:
# fix các lỗi trên

# Thay thế số 0 bằng NaN cho các cột bị lỗi
df['runtime'] = df['runtime'].replace(0, np.nan)
df['revenue'] = df['revenue'].replace(0, np.nan)

# Riêng vote_average bằng 0 nghĩa là "chưa có vote", lọc bỏ khi phân tích điểm số

In [9]:
# Xem thử phim nào dài 21 tiếng
print(df[df['runtime'] > 300][['title', 'runtime']])

                               title  runtime
6747                           Shoah    566.0
7415       Tinker Tailor Soldier Spy    320.0
7420               The Best of Youth    366.0
7933                      The Deluge    315.0
8635                   War and Peace    422.0
...                              ...      ...
44176                      The Idiot    550.0
44464  The Guyver: Bio-Booster Armor    360.0
44721                    The Keepers    432.0
44758                      Hollywood    780.0
45462            Century of Birthing    360.0

[108 rows x 2 columns]


In [10]:
# Lọc ra chính xác bộ phim có thời lượng dài nhất
longest_movie = df[df['runtime'] == df['runtime'].max()]
print(longest_movie[['title', 'runtime', 'release_date']])

            title  runtime release_date
24178  Centennial   1256.0   1978-10-01


In [11]:
# Xem số lượng dòng trước khi lọc
print(f"Số lượng phim ban đầu: {df.shape[0]}")

# Giữ lại phim có thời lượng từ 45 phút trở lên (loại bỏ phim ngắn, phim lỗi 0 phút)
# Giữ lại phim có thời lượng từ 300 phút trở xuống (loại bỏ 108 phim siêu dài dính outlier)
df_clean = df[(df['runtime'] >= 45) & (df['runtime'] <= 300)].copy()

# Reset lại index của DataFrame
df_clean = df_clean.reset_index(drop=True)

# Kiểm tra lại sau khi lọc
print(f"Số lượng phim sau khi dọn dẹp: {df_clean.shape[0]}")
print(f"Đã loại bỏ được: {df.shape[0] - df_clean.shape[0]} bộ phim lỗi thời lượng.")

# Check lại min, max của dataframe mới
print("\nBảng thống kê thời lượng mới:")
print(df_clean['runtime'].describe())

Số lượng phim ban đầu: 45436
Số lượng phim sau khi dọn dẹp: 41900
Đã loại bỏ được: 3536 bộ phim lỗi thời lượng.

Bảng thống kê thời lượng mới:
count    41900.000000
mean        99.595967
std         22.959280
min         45.000000
25%         88.000000
50%         96.000000
75%        108.000000
max        300.000000
Name: runtime, dtype: float64


In [12]:
# Check status fim
print(df['status'].unique())

# check số lượng fim trước khi lọc status
print(f"Số lượng phim trước khi lọc status: {df_clean.shape[0]}")

# Chỉ giữ lại những dòng có status chính xác là 'Released'
df_clean = df_clean[df_clean['status'] == 'Released'].copy()

# reset_index sau khi lọc để tránh lệch ma trận Embedding của BERT
df_clean = df_clean.reset_index(drop=True)

# Kiểm tra lại số lượng phim sau khi lọc
print(f"Số lượng phim sau khi dọn dẹp status: {df_clean.shape[0]}")
print(f"Mảng các giá trị độc nhất bây giờ: {df_clean['status'].unique()}")

['Released' nan 'Rumored' 'Post Production' 'In Production' 'Planned'
 'Canceled']
Số lượng phim trước khi lọc status: 41900
Số lượng phim sau khi dọn dẹp status: 41542
Mảng các giá trị độc nhất bây giờ: ['Released']


In [13]:
# vì genres đang là 1 list nên cần trải ra + clean
import ast

# Hàm giải mã chuỗi và trích xuất chỉ lấy tên + thể loại
def extract_genres(genres_str):
    # Nếu dòng Null/NaN -> trả về chuỗi trống
    if pd.isna(genres_str) or genres_str == "":
        return ""

    try:
        # ast.literal_eval biến chuỗi "[{'id': 16, 'name': 'Animation'}]" thành list
        genres_list = ast.literal_eval(genres_str)

        # Lấy giá trị của key 'name' trong từng dictionary và nối lại bằng dấu phẩy
        names = [genre['name'] for genre in genres_list]
        return ", ".join(names)
    except:
        # Phòng trường hợp dòng nào đó dính lỗi định dạng đặc biệt
        return ""

# 2. Áp dụng hàm lên cột genres của df_clean
df_clean['genres'] = df_clean['genres'].apply(extract_genres)

# 3. Xem thử thành quả sau khi dọn dẹp sạch cột genres
print("Kiểu dữ liệu mới của ô đầu tiên:", type(df_clean['genres'].iloc[0]))
print("\nXem thử vài dòng dữ liệu genres sau khi làm sạch:")
print(df_clean[['title', 'genres']].head())

Kiểu dữ liệu mới của ô đầu tiên: <class 'str'>

Xem thử vài dòng dữ liệu genres sau khi làm sạch:
                         title                      genres
0                    Toy Story   Animation, Comedy, Family
1                      Jumanji  Adventure, Fantasy, Family
2             Grumpier Old Men             Romance, Comedy
3            Waiting to Exhale      Comedy, Drama, Romance
4  Father of the Bride Part II                      Comedy


In [14]:
# check overview fim (mô tả fim)
df_clean['overview'] = df_clean['overview'].fillna('')

# Xóa các câu text hệ thống tự điền khi thiếu mô tả -> chuyển về chữ thường
garbage_phrases = [
    'no overview found',
    'no overview available',
    'no plot text',
    'no plot available'
]

# Hàm dọn rác text mô tả
def clean_overview_text(text):
    if text.strip().lower() in garbage_phrases:
        return ""
    return text

df_clean['overview'] = df_clean['overview'].apply(clean_overview_text)

# check thử
empty_overviews = (df_clean['overview'] == "").sum()
print(f"Số lượng phim không có mô tả (đã đưa về chuỗi trống): {empty_overviews}")

Số lượng phim không có mô tả (đã đưa về chuỗi trống): 274


In [ ]:
# export file data đã clean
df_clean.to_csv("movie_dataset_cleaned.csv", index=False, encoding="utf-8-sig")

# Feature Extraction

In [16]:
# Các cột được giữ lại và dùng cho mô hình
selected_features = [
    'id',
    'title',
    'genres',
    'overview',
    'release_date',
    'runtime',
    'popularity',
    'vote_average'
]

# Trích xuất DataFrame mới chỉ giữ lại các cột trên
df_final = df_clean[selected_features].copy()

# Add release_year to df_final
df_final['release_year'] = pd.to_datetime(df_final['release_date'], errors='coerce').dt.year

# Kiểm tra lại hình dạng của dữ liệu mới
print("Kích thước dataset sau khi chọn lọc feature:", df_final.shape)
print("\n5 dòng dữ liệu mẫu cuối cùng sẵn sàng cho các bước sau:")
print(df_final.head())

Kích thước dataset sau khi chọn lọc feature: (41542, 9)

5 dòng dữ liệu mẫu cuối cùng sẵn sàng cho các bước sau:
      id                        title                      genres  \
0    862                    Toy Story   Animation, Comedy, Family   
1   8844                      Jumanji  Adventure, Fantasy, Family   
2  15602             Grumpier Old Men             Romance, Comedy   
3  31357            Waiting to Exhale      Comedy, Drama, Romance   
4  11862  Father of the Bride Part II                      Comedy   

                                            overview release_date  runtime  \
0  Led by Woody, Andy's toys live happily in his ...   1995-10-30     81.0   
1  When siblings Judy and Peter discover an encha...   1995-12-15    104.0   
2  A family wedding reignites the ancient feud be...   1995-12-22    101.0   
3  Cheated on, mistreated and stepped on, the wom...   1995-12-22    127.0   
4  Just when George Banks has recovered from his ...   1995-02-10    106.0   

  p

## Gộp các trường thông tin lại

Vì BERT chỉ nhận vào 1 chuỗi văn bản duy nhất cho mỗi đối tượng (input), không thể nhận một lúc 3-4 cột riêng biệt

Tạo một vector không gian duy nhất (single vector space) -> 1 vector (384 chiều) đại diện cho đúng bộ fim, khi user muốn tìm fim tương tự -> so sánh vector fim A với fim B

In [18]:
from sentence_transformers import SentenceTransformer
import torch

# 1. Gộp các thông tin từ Feature Selection
df_final['movie_content'] = (
    "Tên phim: " + df_final['title'].fillna('') +
    " . Thể loại: " + df_final['genres'].fillna('') +
    " . Nội dung: " + df_final['overview'].fillna(''))

print("--- KIỂM TRA INPUT SAU KHI GỘP FEATURE ---")
print(df_final['movie_content'].iloc[0])
print("-" * 50)

--- KIỂM TRA INPUT SAU KHI GỘP FEATURE ---
Tên phim: Toy Story . Thể loại: Animation, Comedy, Family . Nội dung: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.
--------------------------------------------------


# Applying BERT

## BERT Preprocessing (Tokenize, Embedding, etc)

- Model dùng: all-MiniLM-L6-v2 (pretrained)

- Vì dùng model trên nên preprocessing không quá phức tạp, chỉ cần gọi hàm model.encode(). Hàm tự động chạy ngầm các bước sau:

- Tokenize: Tách câu -> các từ đơn ->  chuyển văn bản thành các ID số mà BERT hiểu.

- Padding & Truncation: Tự cắt ngắn nếu văn bản dài quá 256/512 tokens hoặc thêm chuỗi trống nếu quá ngắn để đưa về độ dài bằng nhau.

- Forward Pass: Đưa qua các tầng Transformer để lấy kết quả nhúng thô.

- Pooling (Trích xuất Embedding cuối cùng): Gom các vector của từng từ lại thành một vector duy nhất đại diện cho cả bộ phim (thường dùng Mean Pooling).

In [19]:
# import BERT model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
# Kiểm tra độ dài tối đa mà mô hình hỗ trợ xử lý cho một chuỗi text
print("Độ dài chuỗi tối đa mô hình hỗ trợ (max_seq_length):", model.get_max_seq_length())

Độ dài chuỗi tối đa mô hình hỗ trợ (max_seq_length): 256


all-miniLM-L6-v2 là Sentence-BERT bi encoder, phiên bản nhẹ hơn của BERT, được train bằng knowledge distillation, ít layer hơn nên train nhanh hơn.

- L6 = 6 transformer layers
- BERT-base = 12 layers
-> nhanh hơn

- v2: được fine-tune cho semantic similarity + retrieval, tối ưu cosine similarity

Input đưa vào là 1 câu hoặc 1 đoạn text, ở đây là tên phim. Output là 1 vector embedding (384 dims) trong khi BERT-base là 768 dims.
Các câu (tìm kiếm của user và tên phim tương ứng) sẽ có vector gần nhau.

Ví dụ:
Query 1 = "Avengers: Endgame"
Query 2 = "Avengers Infinity War"

-> Sau khi encode, vector của hai phim sẽ gần nhau vì chúng thuộc cùng series, có nội dung và nhân vật liên quan. Do đó cosine similarity giữa hai vector sẽ cao.

In [21]:
# Tokenize + embedding + pooling ngầm
print("Đang thực hiện BERT preprocessing và áp dụng Bi-encoder...")

movie_embeddings = model.encode(
    df_final['movie_content'].tolist(),
    show_progress_bar=True,
    convert_to_tensor=True
)

print("Quá trình Preprocessing và  áp dụng Bi-encoder hoàn tất!")

Đang thực hiện BERT preprocessing và áp dụng Bi-encoder...


Batches:   0%|          | 0/1299 [00:00<?, ?it/s]

Quá trình Preprocessing và  áp dụng Bi-encoder hoàn tất!


In [22]:
# Check hình dáng của ma trận vector
print("Kích thước ma trận embedding:", movie_embeddings.shape)

Kích thước ma trận embedding: torch.Size([41542, 384])


In [23]:
# Xem thử vector đại diện của bộ phim đầu tiên (Toy Story)
# Lấy dòng index 0, in ra 10 số đầu tiên trong tổng số 384 chiều của nó
print("10 giá trị đặc trưng đầu tiên của phim Toy Story:")
print(movie_embeddings[0][:10])

10 giá trị đặc trưng đầu tiên của phim Toy Story:
tensor([ 0.0081, -0.0476,  0.0831, -0.0752,  0.0477,  0.0428,  0.1333,  0.0070,
        -0.0197, -0.0096])


In [24]:
# Check nan/null trong vector
# Nếu movie_embeddings là PyTorch Tensor (mặc định khi dùng convert_to_tensor=True)
has_nan = torch.isnan(movie_embeddings).any().item()
print("Ma trận vector có bị dính lỗi NaN (mất dữ liệu) không?:", has_nan)

Ma trận vector có bị dính lỗi NaN (mất dữ liệu) không?: False


In [25]:
# Check thử BERT = 3 fim đầu tiên

from sentence_transformers import util

# Tính thử độ tương đồng giữa Toy Story (hoạt hình gia đình) vs Jumanji (phiêu lưu giả tưởng)
sim_0_1 = util.cos_sim(movie_embeddings[0], movie_embeddings[1]).item()

# Tính thử độ tương đồng giữa Toy Story với Grumpier Old Men (tình cảm người lớn)
sim_0_2 = util.cos_sim(movie_embeddings[0], movie_embeddings[2]).item()

print(f"Độ tương đồng giữa Toy Story và Jumanji: {sim_0_1:.4f}")
print(f"Độ tương đồng giữa Toy Story và Grumpier Old Men: {sim_0_2:.4f}")

Độ tương đồng giữa Toy Story và Jumanji: 0.6925
Độ tương đồng giữa Toy Story và Grumpier Old Men: 0.5934


In [26]:
# Chuẩn hóa (normalize)
import torch

# Chuẩn hóa L2 cho ma trận vector
# Sau khi chuẩn hóa, bình phương độ dài của mỗi vector phim sẽ bằng 1
movie_embeddings_norm = torch.nn.functional.normalize(movie_embeddings, p=2, dim=1)

print("Kích thước ma trận vector sau chuẩn hóa:", movie_embeddings_norm.shape)

Kích thước ma trận vector sau chuẩn hóa: torch.Size([41542, 384])


# Calculate cosine similarity

In [53]:
from sentence_transformers import util

def recommend_movies(movie_title, top_k=5):
    # Tìm index của bộ phim người dùng nhập vào trong df_final
    movie_title_clean = movie_title.strip().lower()
    matched_movies = df_final[df_final['title'].str.lower().str.contains(movie_title_clean)]

    if matched_movies.empty:
        print(f"Không tìm thấy phim '{movie_title}' trong hệ thống dữ liệu!")
        return None

    # Lấy index dòng đầu tiên tìm được
    idx = matched_movies.index[0]
    # df_final = df_final.reset_index(drop=True)

    # 2. Lấy vector đặc trưng của bộ phim mục tiêu này
    target_embedding = movie_embeddings_norm[idx]

    # 3. Tính cosine similarity động
    # Tính độ tương đồng giữa 1 vector phim này với toàn bộ ma trận 41.542 vector phim kia
    cosine_scores = util.cos_sim(target_embedding, movie_embeddings_norm)[0]

    # 4. Candidate movie retrieval (lọc top-k)
    # Sắp xếp điểm số từ cao xuống thấp và lấy ra top_k + 1 phim (cộng 1 vì phim giống nhất là chính nó)
    top_results = torch.topk(cosine_scores, k=top_k + 1)

    # Lấy ra index và điểm số tương đồng thực tế
    top_indices = top_results.indices.cpu().numpy()
    top_scores = top_results.values.cpu().numpy()

    # Loại bỏ vị trí đầu tiên (vì index đầu tiên luôn là chính bộ phim đó với điểm Cosine = 1.0)
    recommended_indices = top_indices[1:]
    recommended_scores = top_scores[1:]

    # 5. Trích xuất kết quả xuất ra bảng
    recommendations = df_final.iloc[recommended_indices].copy()
    recommendations['similarity_score'] = recommended_scores

    print(f"Vì bạn đã xem phim: '{df_final.iloc[idx]['title']}'")
    print(f"Hệ thống gợi ý Top {top_k} bộ phim có nội dung tương đồng nhất sau đây:\n")

    # Hiển thị các trường thông tin quan trọng ra màn hình
    return recommendations[['title', 'genres', 'release_date', 'vote_average', 'similarity_score']]

In [57]:
# test
movie_name = input("Nhập tên phim: ")

results = recommend_movies(movie_name, top_k=10)

if results is not None:
    print("\n===== DANH SÁCH PHIM ĐỀ XUẤT =====\n")

    for i, row in enumerate(results.itertuples(), start=1):
        print(f"{i}. {row.title}")
        print(f"   Thể loại: {row.genres}")
        print(f"   Năm phát hành: {row.release_date}")
        print(f"   Rating: {row.vote_average}")
        print(f"   Similarity: {row.similarity_score:.4f}")
        print("-" * 50)

Nhập tên phim: The Dark Tapes
Vì bạn đã xem phim: 'The Dark Tapes'
Hệ thống gợi ý Top 10 bộ phim có nội dung tương đồng nhất sau đây:


===== DANH SÁCH PHIM ĐỀ XUẤT =====

1. Tales from the Crapper
   Thể loại: Science Fiction, Horror, Comedy
   Năm phát hành: 2004-01-29
   Rating: 3.6
   Similarity: 0.8507
--------------------------------------------------
2. Demonic
   Thể loại: Thriller, Horror
   Năm phát hành: 2015-02-12
   Rating: 5.0
   Similarity: 0.8502
--------------------------------------------------
3. Night of Dark Shadows
   Thể loại: Drama, Horror, Thriller, Mystery, Romance, Foreign
   Năm phát hành: 1971-08-04
   Rating: 6.0
   Similarity: 0.8485
--------------------------------------------------
4. Paranormal Activity: The Ghost Dimension
   Thể loại: Horror, Thriller
   Năm phát hành: 2015-10-21
   Rating: 5.0
   Similarity: 0.8471
--------------------------------------------------
5. The Ghost
   Thể loại: Drama, Thriller
   Năm phát hành: 2008-11-13
   Rating: 5.9